# Seed comparison: synthetic vs LLM vs ESConv

Purpose: run a tiny grid to compare the three seeding modes before choosing one for larger Phase 3 runs.

- **Synthetic seeds**: fixed templates per emotion (controlled, but less realistic).
- **LLM seeds**: generated per trial (more variety; can drift or be empty if model is flaky).
- **ESConv seeds**: sampled utterances from ESConv, classifier-checked to match target emotion (most realistic; requires ESConv download).

Outputs: three CSVs/heatmaps under `results/seed_comparison_*` plus meta/log files.

In [ ]:
from pathlib import Path
import json
from datasets import load_dataset
from dynamic_conversation import (
    SingleTurnSimulator,
    SimulationConfig,
    ResponseStrategy,
    build_esconv_seed_bank,
)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

# Shared config
cfg = SimulationConfig(
    model="gpt-4o-mini",
    max_tokens=800,
    temperature=1.0,
    brevity_hint="Reply in 1-2 sentences, keep emotion visible."
)

emotions = ["anger", "joy"]
strategies = [ResponseStrategy.VALIDATE, ResponseStrategy.GUIDE]
style_mod = "concise and emotionally attuned"

# Build ESConv seed bank (classifier-checked into DistilRoBERTa labels)
esconv = load_dataset("thu-coai/esconv")
seed_bank = build_esconv_seed_bank(esconv, emotions=emotions, per_emotion=6, max_chars=200, use_gpu=False)
seed_bank

In [ ]:
def run_and_report(suffix, use_llm_seed=False, use_esconv_seed=False, esconv_seeds=None):
    sim = SingleTurnSimulator(
        use_gpu=False,
        config=cfg,
        prompt_for_key=True,
        esconv_seeds=esconv_seeds,
    )
    df = sim.run_batch(
        emotions=emotions,
        strategies=strategies,
        runs_per_pair=1,
        style_modifier=style_mod,
        use_llm_seed=use_llm_seed,
        include_baseline=True,
        use_esconv_seed=use_esconv_seed,
        save_csv=results_dir / f"seed_comparison_{suffix}.csv",
        save_heatmap=results_dir / f"seed_comparison_{suffix}_heatmap.png",
    )
    meta_path = results_dir / f"seed_comparison_{suffix}.meta.json"
    log_path = results_dir / f"seed_comparison_{suffix}.log"
    meta = json.load(open(meta_path)) if meta_path.exists() else {}
    log = log_path.read_text() if log_path.exists() else ""
    return df, meta, log

# Synthetic seeds (default)
df_syn, meta_syn, log_syn = run_and_report("synthetic", use_llm_seed=False, use_esconv_seed=False)

# LLM seeds
df_llm, meta_llm, log_llm = run_and_report("llm", use_llm_seed=True, use_esconv_seed=False)

# ESConv seeds
df_esconv, meta_esconv, log_esconv = run_and_report("esconv", use_llm_seed=False, use_esconv_seed=True, esconv_seeds=seed_bank)


In [ ]:
print("Synthetic meta:", meta_syn)
print("Synthetic log:\n", log_syn)

print("LLM meta:", meta_llm)
print("LLM log:\n", log_llm)

print("ESConv meta:", meta_esconv)
print("ESConv log:\n", log_esconv)

df_syn, df_llm, df_esconv